In [17]:
import pandas as pd
import torch

# Per riproducibilità
torch.manual_seed(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("../data/all_ships.csv")


In [18]:
from sklearn.model_selection import train_test_split

X = df[["sex", "age", "age2", "class", "sex_class", "crew"]]
Y = df["survived"]

X_tensor = torch.tensor(X.values, dtype=torch.float32)
Y_tensor = torch.tensor(Y.values, dtype=torch.float32) # la BCEWithLogitsLoss richiede target float

# per riproducibilità si usa random_state fissato
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=42, stratify=Y_tensor)

mean = X_train.mean(0)
std = X_train.std(0)

X_train_norm = (X_train - mean) / std
X_test_norm = (X_test - mean) / std


In [19]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_norm, Y_train)
test_ds = TensorDataset(X_test_norm, Y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [20]:
from torch import nn

class LogisticRegressor(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, out_features=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [21]:
class LogisticRegressorLogits(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features=1)

    def forward(self, x):
        return self.linear(x)

In [22]:
from torch.optim import SGD
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score

def train_model(model, train_loader, test_loader, criterion, lr=0.05, epochs=300):
    writer = SummaryWriter(f'../results/{model._get_name()}')
    model = model.to(device)

    optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=0.001)

    use_logits = isinstance(criterion, nn.BCEWithLogitsLoss)

    for epoch in range(epochs):
        model.train()

        train_loss = 0.0
        y_true = []
        y_pred = []
        
        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            output = model(X_batch)
            loss = criterion(output.view(-1), Y_batch) # output e Y_batch devono avere stesso shape (batch_size,)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            train_loss += loss.item() * X_batch.size(0)

            if use_logits:
                # l'output sono logits quindi li normalizzo a probabilità con la sigmoide e uso la soglia per scegliere la classe
                probs = torch.sigmoid(output.view(-1))
            else:
                probs = output

            preds = (probs >= 0.5).int() 
            y_true.extend(Y_batch.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

        train_loss /= len(train_loader.dataset)
        train_acc = accuracy_score(y_true, y_pred)

        model.eval()

        test_loss = 0.0
        y_true = []
        y_pred = []

        with torch.no_grad():
            for X_batch, Y_batch in test_loader:
                X_batch = X_batch.to(device)
                Y_batch = Y_batch.to(device)

                output = model(X_batch)
                loss = criterion(output.view(-1), Y_batch) # output e Y_batch devono avere stesso shape (batch_size,)
                test_loss += loss.item() * X_batch.size(0)

                if use_logits:
                    # l'output sono logits quindi li normalizzo a probabilità con la sigmoide e uso la soglia per scegliere la classe
                    probs = torch.sigmoid(output.view(-1))
                else:
                    probs = output.view(-1)

                preds = (probs >= 0.5).int() 
                y_true.extend(Y_batch.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())

        test_loss /= len(test_loader.dataset)
        test_acc = accuracy_score(y_true, y_pred)

        writer.add_scalar('loss/train', train_loss, epoch)
        writer.add_scalar('accuracy/train', train_acc, epoch)
        writer.add_scalar('loss/test', test_loss, epoch)
        writer.add_scalar('accuracy/test', test_acc, epoch)

        if epoch % 50 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")

    print(f"Epoch {epoch+1}/{epochs} | train_loss {train_loss:.4f} | train_acc {train_acc:.4f} | test_loss {test_loss:.4f} | test_acc {test_acc:.4f}")
    writer.close()

    return model, train_loss, train_acc, test_loss, test_acc

In [23]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_model(model, X_test_norm, Y_test, criterion):
    model.eval()
    X_test_norm = X_test_norm.to(device)

    with torch.no_grad():
        output = model(X_test_norm)

        # se il modello restituisce logits applico sigmoid
        if isinstance(criterion, nn.BCEWithLogitsLoss):
            probs = torch.sigmoid(output.view(-1))
        else:
            probs = output.view(-1)

        y_pred = (probs >= 0.5).int()

    y_true = Y_test.cpu().numpy()
    y_pred = y_pred.cpu().numpy()

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

In [24]:
model = LogisticRegressor(in_features=6)
criterion = nn.BCELoss()
model, train_loss, train_acc, test_loss, test_acc = train_model(model, train_loader, test_loader, criterion)
evaluate_model(model, X_test_norm, Y_test, criterion)

model = LogisticRegressorLogits(in_features=6)
# uso pos_weight poichè il dataset è sbilanciato cerco di renderlo più bilanciato
counts = torch.bincount(Y_train.view(-1).to(torch.int64))
pos_weight = (counts[0] / counts[1]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) 
model, train_loss, train_acc, test_loss, test_acc = train_model(model, train_loader, test_loader, criterion)
evaluate_model(model, X_test_norm, Y_test, criterion)

Epoch 1/300 | train_loss 0.6034 | train_acc 0.6862 | test_loss 0.5903 | test_acc 0.6879
Epoch 51/300 | train_loss 0.5778 | train_acc 0.7090 | test_loss 0.5833 | test_acc 0.6762
Epoch 101/300 | train_loss 0.5755 | train_acc 0.7096 | test_loss 0.5691 | test_acc 0.7100
Epoch 151/300 | train_loss 0.5763 | train_acc 0.7148 | test_loss 0.5713 | test_acc 0.7113
Epoch 201/300 | train_loss 0.5746 | train_acc 0.7096 | test_loss 0.5709 | test_acc 0.7126
Epoch 251/300 | train_loss 0.5735 | train_acc 0.7142 | test_loss 0.5646 | test_acc 0.7126
Epoch 300/300 | train_loss 0.5755 | train_acc 0.7106 | test_loss 0.5756 | test_acc 0.7152
Accuracy : 0.7152
Precision : 0.8605
Recall : 0.1480
F1 Score : 0.2526
Epoch 1/300 | train_loss 0.9029 | train_acc 0.5840 | test_loss 0.8663 | test_acc 0.5709
Epoch 51/300 | train_loss 0.8591 | train_acc 0.5934 | test_loss 0.8617 | test_acc 0.6957
Epoch 101/300 | train_loss 0.8590 | train_acc 0.5866 | test_loss 0.8403 | test_acc 0.5462
Epoch 151/300 | train_loss 0.8619 |